In [163]:
from tqdm import tqdm
import json
import os

from google import genai
from google.genai import types
from pydantic import BaseModel, Field


In [164]:
path_database = "nuovo_database_sentenze_target.json"

with open(path_database, 'r', encoding='utf-8') as j:
    database = json.load(j)

In [165]:
sys = """ Sei un estrattore giuridico specializzato. Il tuo compito è leggere frammenti di sentenze civili italiane, estrarre tutti i riferimenti normativi e giurisprudenziali presenti e restituirli ESCLUSIVAMENTE in un formato JSON strutturato.

Devi normalizzare i riferimenti estratti per creare i nodi di un Knowledge Graph, inserendoli nella chiave "nodo_grafo". Applica rigorosamente SOLO le seguenti regole di formattazione:

1. SENTENZE DELLA CASSAZIONE
Ignora sezione, giorno, mese e tipo (sentenza/ordinanza). Estrai solo numero e anno.
Formato: Cass. Civ. [numero]/[anno]
Esempio: "Cass. Sez. 3, Sentenza n. 1057 del 10/04/1956" -> Cass. Civ. 1057/1956

2. MASSIMARIO (REPERTORIO)
Ignora le date. Estrai solo il numero a 6 cifre.
Formato: Rv. [numero]
Esempio: "del 16/06/2004, Rv. 573669" -> Rv. 573669

3. CODICI
Ignora i commi. Usa "Art." seguito dal numero e abbreviazione del codice (c.c., c.p.c., c.p., cod. cons.).
Formato: Art. [numero] [abbreviazione_codice]
Esempio: "all'art. 327 c.p.c., primo comma" -> Art. 327 c.p.c.

4. LEGGI, DECRETI E REGI DECRETI (INTERI)
Ignora giorno e mese. Mantieni la sigla ufficiale (L., D.Lgs., D.L., D.P.R., R.D.), numero e anno a 4 cifre.
Formato: [Tipo] [numero]/[anno]
Esempio: "legge 7 agosto 1990 n. 241" -> L. 241/1990

5. ARTICOLI DI LEGGI O DECRETI
Ignora i commi. Combina articolo e fonte normativa.
Formato: Art. [numero] [Tipo] [numero]/[anno]
Esempio: "articolo 33 del D.Lgs 206/05" -> Art. 33 D.Lgs. 206/2005

6. COSTITUZIONE
Formato: Art. [numero] Cost.
Esempio: "art. 24 della Costituzione" -> Art. 24 Cost.

7. DIRITTO DELL'UNIONE EUROPEA
Per Direttive e Regolamenti (CE, CEE, UE), usa solo 'Dir. UE' o 'Reg. UE'. L'anno a 4 cifre va sempre prima della barra.
Formato: [Dir. UE / Reg. UE] [anno]/[numero]
Esempio: "Regolamento (CE) n. 261/2004" -> Reg. UE 2004/261

8. REGOLA DI FALLBACK
Se un riferimento (es. Circolari, Contratti Collettivi, normative locali) non rientra in NESSUNO dei 7 casi, NON inventare formati. Restituisci la stringa estratta così come si presenta, rimuovendo solo gli articoli prepositivi iniziali e gli spazi in eccesso.
"""

In [166]:
class Riferimento(BaseModel):
    testo: str = Field(description='il testo esatto estratto dall\'input')
    nodo: str = Field(description='il nome standardizzato del riferimento')

class Chunk(BaseModel):
    chunk_id: int = Field(description='l\'id del chunk')
    riferimenti: list[Riferimento]

class Sentenza(BaseModel):
    sentenza_id: str = Field(description='l\'id della sentenza')
    chunks: list[Chunk] = Field(description='Lista dei chunk. Anche se i chunk non contengono riferimenti, questa sentenza deve essere sempre inclusa nell\'output con il suo ID.')

class Batch(BaseModel):
    sentenze:list[Sentenza]


In [ ]:
riferimenti_normativi = []
batch = []

for sentenza in tqdm(database):

    chunks = [sentenza['chunks'][c]['chunk'] for c in sentenza['chunks'].keys()]
    sentenza_id = sentenza['ID']

    if len(batch) < 6 and database[-1] != sentenza:

        richiesta = f'SENTENZA ID: {sentenza_id}\n'
        for ind, c in enumerate(chunks):
            richiesta += f'chunk n°{ind}:\n {c}'
        
        batch.append(richiesta)

    elif len(batch) < 7 and database[-1] == sentenza:

        richiesta = f'SENTENZA ID: {sentenza_id}\n'
        for ind, c in enumerate(chunks):
            richiesta += f'chunk n°{ind}:\n {c}\n'
        
        batch.append(richiesta)

        client = genai.Client(api_key='key')
        output = client.models.generate_content(
            model='gemini-3.1-flash-lite',
            contents = '\n'.join(batch),
            config = types.GenerateContentConfig(
                temperature=0,
                system_instruction= sys,
                response_schema=Batch,
                response_mime_type='application/json'
            ))
        
        batch_oggetti = Batch.model_validate_json(output.text)
        riferimenti_normativi.extend(batch_oggetti.sentenze)

    elif len(batch) == 6:

        richiesta = f'SENTENZA ID: {sentenza_id}\n'
        for ind, c in enumerate(chunks):
            richiesta += f'chunk n°{ind}:\n {c}'
        
        batch.append(richiesta)

        client = genai.Client(api_key='key')
        output = client.models.generate_content(
            model='gemini-3.1-flash-lite',
            contents = '\n'.join(batch),
            config = types.GenerateContentConfig(
                temperature=0,
                system_instruction= sys,
                response_schema=Batch,
                response_mime_type='application/json'
            ))
        
        batch = []
        batch_oggetti = Batch.model_validate_json(output.text)
        riferimenti_normativi.extend(batch_oggetti.sentenze)

        


100%|██████████| 32/32 [02:42<00:00,  5.08s/it]


In [ ]:
if os.path.exists('riferimenti_normativi.json'):

    with open('riferimenti_normativi.json', 'r', encoding='utf-8') as j:
        rif = json.load(j)

    rif.extend([item.model_dump() for item in riferimenti_normativi])

else:

    rif = [item.model_dump() for item in riferimenti_normativi]

with open('riferimenti_normativi.json', 'w', encoding='utf-8') as rn:
        json.dump(rif, rn, ensure_ascii=False, indent=4)


